In [ ]:
import math
import numpy as np

activations = []
weighted_sums = []
beta_1 = 0.90
beta_2 = 0.999
learning_rate = 0.001
moment_1_history = {}
moment_2_history = {}
epsilon = 10**-13

In [ ]:
def sparse_categorical_crossentropy(prediction_probabilities_map, true_class):
    true_class_prediction = prediction_probabilities_map.get(true_class, 0)
    return -1*(math.log(true_class_prediction))

In [ ]:
def relu(pred):
    return max(0,pred)

vectorized_relu = np.vectorize(relu)

In [ ]:
def softmax(output_layer_output_map):
    output_layer_keys_softmax_map = {key:0 for key in output_layer_output_map.keys()}
    sum_of_euler_powers = 0
    for key,val in output_layer_output_map.items():
        sum_of_euler_powers+= (math.e)**val
    
    for key,val in output_layer_output_map.items():
        output_layer_keys_softmax_map[key] = ((math.e)**val)/sum_of_euler_powers
    
    return output_layer_keys_softmax_map

vectorized_softmax = np.vectorize(softmax)

In [ ]:
def softmax_sparse_categorical_crossentropy_gradient(logits_prediction_arr, true_class_idx):
    logits_prediction_arr[true_class_idx]-=1
    return logits_prediction_arr

In [ ]:
def output_layers_weight_updates(previous_layers_activations, neurons_predicted_outputs):
    multiplier_for_probs = softmax_sparse_categorical_crossentropy_gradient(neurons_predicted_outputs)    
    change_for_weights = np.outer(previous_layers_activations, multiplier_for_probs)
    return change_for_weights

In [ ]:
def relu_derivative(neurons_weighted_sum):
    return 1 if neurons_weighted_sum>0 else 0
vectorized_relu_derivative = np.vectorize(relu_derivative)

In [ ]:
def hidden_layer_backprop_gradients_for_one_neuron(curr_activations, next_layers_error_gradients, next_layers_weights):
    derivatives_of_curr_activations = vectorized_relu_derivative(curr_activations)
    delta_array = np.outer(np.outer(next_layers_weights,next_layers_error_gradients),derivatives_of_curr_activations)
    return np.sum(delta_array)


In [ ]:
def adam_optimizer(curr_weights, curr_gradients, iter, layer):
    moment_1_curr = (beta_1*(moment_1_history.get(layer,np.zeros(curr_weights.shape)))) + (1-beta_1) * curr_gradients
    moment_1_history[layer] = moment_1_curr

    hat_moment_1 = moment_1_curr/(1-beta_1**iter)

    moment_2_curr = beta_2*(moment_2_history.get(layer,np.zeros(curr_weights.shape))) + (1-beta_2) * curr_gradients**2
    moment_2_history[layer] = moment_2_curr

    hat_moment_2 = moment_2_curr/(1-beta_2**iter)

    new_weights = curr_weights - ((learning_rate/(np.sqrt(hat_moment_2)+epsilon)) * hat_moment_1)

    return new_weights

In [ ]:
def forward_propagation(input_layer,weights, biases):
    input_matrix = input_layer
    prev_activations = input_matrix
    prediction = None
    for index in range(len(weights)):
        curr_weights = weights[index]
        weighted_sum = prev_activations @ curr_weights
        weighted_sums.append(weighted_sum)
        bias_added_weights = weighted_sum + biases[index]
        if index < len(weights)-1:
            activations.append(prev_activations)
            prev_activations = vectorized_relu(bias_added_weights)
        else:
            activations.append(prev_activations)
            prediction = vectorized_softmax(bias_added_weights)
            
    return prediction
        
        





In [ ]:
def backward_propagation(layer_weights, layer_biases, prediction_array, batch_true_labels, time_step):

    # backprop for output layer

    output_layer_error_gradients = softmax_sparse_categorical_crossentropy_gradient(prediction_array,batch_true_labels)
    output_layer_weights_change = output_layer_error_gradients*activations[-2]
    # adam_optimized_output_layer_weights_change = adam_optimizer(layer_weights[-1],output_layer_weights_change,time_step,len(layer_weights)-1)


    # adam_optimized_weights_change_for_layers = {}

    # adam_optimized_weights_change_for_layers[len(layer_weights)-1] = adam_optimized_output_layer_weights_change

    errors_for_backprop = {
        len(layer_weights)-1:output_layer_weights_change
    }

    # backprop for the rest

    for l in range(len(layer_weights)-2,-1,-1):
        delta_j_arr = [0 for i in range(len(layer_weights[l]))]
        for neuron  in range(len(layer_weights[l])):
            delta_j_arr[neuron] = hidden_layer_backprop_gradients_for_one_neuron(activations[l],errors_for_backprop[l+1],layer_weights[l+1])
        errors_for_backprop[l] = delta_j_arr
    
    #apply adam

    for l in range(len(layer_weights)):
        layer_changes = errors_for_backprop[l]
        adam_optimized_changes = adam_optimizer(layer_weights[l],layer_changes,time_step,l)
        layer_weights -= adam_optimized_changes






In [ ]:
def main_loop():
    layer_weights = [
        np.random.rand(764,128),
        np.random.rand(128,64),
        np.random.rand(64,32),
        np.random.rand(32,10)
    ]

    layer_biases = [
        np.random.rand(1,128),
        np.random.rand(1,64),
        np.random.rand(1,32),
        np.random.rand(1,10)
    ]

    epochs = 10

    train_data_size = 1200
    batch_data_size = 32

    iters_per_batch = train_data_size // batch_data_size

    for i in range(epochs):
        for j in range(iters_per_batch):
            batch_inputs = []
            batch_true_labels = []
            activations = []
            weighted_sums = []

            prediction_array = forward_propagation(batch_inputs, layer_weights, layer_biases)
            
            time_step = i*j
            backward_propagation(layer_weights, layer_biases, prediction_array, batch_true_labels, time_step)
    
    np.savez("nn_model.npz", weights = layer_weights, biases = layer_biases)
    


            


In [ ]:
data = np.load('nn_model.npz')
loaded_w = data['weights']
loaded_b = data['biases']